# File formats and Hive ELT (ORC, Parquet, RCFILE, Avro, JSON)

Two batch jobs, then SQL over what they wrote:

* `formats`: CSV -> Parquet -> ORC -> JSON -> Avro -> CSV, checking the data survives every hop
* `hive-elt`: Hive-style ELT in Spark SQL: external TEXTFILE table -> ORC -> partitioned
  Parquet (dynamic partitions) -> RCFILE archive -> summary table and view

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
r = lab.run('formats', run_id=RID)
lab.log_tail(r, 'rows')

## Read each format straight from RustFS with SQL
Spark SQL can query a folder directly: ``format.`path` ``.

In [ ]:
import pandas as pd
counts = {}
for fmt in ["parquet", "orc", "json", "avro"]:
    try:
        counts[fmt] = int(lab.sql(f"SELECT count(*) n FROM {fmt}.`s3a://data/output/formats/{fmt}`")["n"][0])
    except Exception as e:
        counts[fmt] = f"error: {str(e)[:80]}"
pd.Series(counts, name="rows")

In [ ]:
lab.s3_ls("data", "output/formats/")

## Hive ELT

In [ ]:
r = lab.run('hive-elt', run_id=RID)
lab.log_tail(r, '[')

In [ ]:
lab.sql("SHOW TABLES IN elt")

## One table per storage format
Check what the metastore recorded for each.

In [ ]:
rows = []
for t in ["raw_orders", "orders_clean", "orders_fact", "orders_archive_rc", "monthly_summary"]:
    d = lab.sql(f"DESCRIBE FORMATTED elt.{t}")
    info = dict(zip(d.iloc[:, 0].str.strip(), d.iloc[:, 1]))
    rows.append({"table": t, "input format": str(info.get("InputFormat", "")).split(".")[-1],
                 "location": info.get("Location")})
pd.DataFrame(rows)

In [ ]:
lab.sql("SHOW PARTITIONS elt.orders_fact")

## The RCFILE archive must match the fact table

In [ ]:
lab.sql("""SELECT (SELECT count(*) FROM elt.orders_fact) fact_rows,
                   (SELECT count(*) FROM elt.orders_archive_rc) rcfile_rows,
                   (SELECT round(sum(amount), 2) FROM elt.orders_fact) fact_revenue,
                   (SELECT round(sum(amount), 2) FROM elt.orders_archive_rc) rcfile_revenue""")

In [ ]:
m = lab.sql("SELECT order_month, region, revenue FROM elt.monthly_summary ORDER BY order_month")
m.pivot(index="order_month", columns="region", values="revenue").astype(float).plot.bar(title="monthly revenue by region");

In [ ]:
lab.sql("SELECT * FROM elt.v_monthly_top WHERE rank_in_month = 1 ORDER BY order_month")